In [ ]:
!pip install pytorch-nlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 1.4 MB/s eta 0:00:00


In [ ]:
!wget --no-check-certificate http://nlp.stanford.edu/data/glove.6B.zip -O /tmp/glove.6B.zip

--2024-06-04 20:03:37--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2024-06-04 20:03:37--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2024-06-04 20:03:37--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘/tmp/glove.6B.zip’


In [ ]:
import numpy as np


import os
import zipfile

from torchtext.transforms import SentencePieceTokenizer
import pandas as pd
import torchnlp
from torchnlp.encoders import Encoder
from torchtext.data import get_tokenizer

with zipfile.ZipFile('/tmp/glove.6B.zip','r') as zip_ref:
  zip_ref.extractall('/tmp/glove')

class PreTrainedEmbeddings():

      def __init__(self):
        self.word_vectors = {}
        f = open('/tmp/glove/glove.6B.100d.txt')
        self.word_to_index = {}
        self.index_to_word = {}
        max_length = 0

        for index,line in enumerate(f):
          values = line.split()
          word = values[0]
          coefs = np.asarray(values[1:], dtype='float32')
          self.word_to_index[word] = index
          self.index_to_word[index] = word
          self.word_vectors[word] = coefs
          max_length = max(max_length,len(coefs))
          #print('Found %s vectors, for %s' % (len(word_vectors), line))
        f.close()

        self.embedding_matrix = np.zeros((len(self.word_to_index)+1, max_length))
        for word, index in self.word_to_index.items():
          embedding_vector = self.word_vectors[word]
          if embedding_vector is not None:
            self.embedding_matrix[index] = embedding_vector
        print('Shape of the matrix: ', self.embedding_matrix.shape)
        self.word_vectors = self.embedding_matrix
        return


      def get_embedding(self, word):
        return self.word_vectors[self.word_to_index[word]]

      def get_closest_to_vector(self, vector, n=1):
        nn_indices = self.index.get_nns_by_vector(vector, n)
        return [self.index_to_word[neighbor] for neighbor in nn_indices]

      def compute_and_print_analogy(self, word1, word2, word3):
        vec1 = self.get_embedding(word1)
        vec2 = self.get_embedding(word2)
        vec3 = self.get_embedding(word3)
        # Simple hypothesis: Analogy is a spatial relationship
        spatial_relationship = vec2 - vec1
        vec4 = vec3 + spatial_relationship
        closest_words = self.get_closest_to_vector(vec4, n=4)
        existing_words = set([word1, word2, word3])
        closest_words = [word for word in closest_words if word not in existing_words]
        if len(closest_words) == 0:
          print("Could not find nearest neighbors for the vector!")
          return
        for word4 in closest_words:
          print("{} : {} :: {} : {}".format(word1, word2, word3,word4))
        return

import gensim.downloader as api

wv = api.load('word2vec-google-news-300')
vec_king = wv['king']

#maybe we should use sentences bcs words wont really help us here, but we can try testing both implementation
from pprint import pprint as print
from gensim.models.fasttext import FastText
from gensim.test.utils import datapath

# Set file names for train and test data
corpus_file = datapath('lee_background.cor')

model = FastText(vector_size=100)

# build the vocabulary
model.build_vocab(corpus_file=corpus_file)

# train the model
model.train(
    corpus_file=corpus_file, epochs=model.epochs,
    total_examples=model.corpus_count, total_words=model.corpus_total_words,
)

print(model)
embeddings = PreTrainedEmbeddings()

AttributeError: 'tuple' object has no attribute 'write'

Start

In [ ]:
!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {"username":"bernardomartins1999","key":"bef461ab78ca866062bdabf72b5271c2"}

import json
import os
import zipfile
from torchtext.transforms import SentencePieceTokenizer
import pandas as pd
import torchnlp
from torchnlp.encoders import Encoder
from torchtext.data import get_tokenizer

with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d jp797498e/twitter-entity-sentiment-analysis

with zipfile.ZipFile('/content/twitter-entity-sentiment-analysis.zip','r') as zip_ref:
  zip_ref.extractall('/content/twitter-entity-sentiment-analysis')

In [ ]:
def cosine(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

In [ ]:
import pandas as pd
import torch

training_data = pd.read_csv('/content/twitter-entity-sentiment-analysis/twitter_training.csv')
validation_data = pd.read_csv('/content/twitter-entity-sentiment-analysis/twitter_validation.csv')

Evaluate data

In [ ]:
#This will be the data used to train the embeddings model
import nltk
from nltk.tokenize import word_tokenize
import numpy as np
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from nltk.corpus import movie_reviews

nltk.download('movie_reviews')
nltk.download('punkt')

documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

tagged_data = [TaggedDocument(words=word_tokenize(' '.join(words).lower()), tags=[category])
              for words, category in documents]

In [ ]:
#there are currently some errors in the data, do some evaluation on it and verify wtf is going on
import re
import matplotlib.pyplot as plt

x_train = training_data.iloc[:, 3].values
y_train = torch.tensor([int(y == 'Positive') for y in training_data.iloc[:, 2].values])

x_test = validation_data.iloc[:, 3].values
y_test = torch.tensor([int(y == 'Positive') for y in validation_data.iloc[:, 2].values])

x_train_df = pd.DataFrame(x_train, columns=['sentence'])
y_train_df = pd.DataFrame(y_train, columns=['sentiment'])

y_train_df.head()


def plot(losses):
  plt.plot([i for i in range(1,len(losses)+1)], losses)

  plt.xlabel('Loss')
  plt.ylabel('Epochs')
  plt.title('Loss function')

  plt.show()

def remove_special_characters(text):
    pattern = r'[^a-zA-Z0-9\s]'  # Matches any character that is not alphanumeric or whitespace
    cleaned_text = re.sub(pattern, '', text)
    return cleaned_text

def get_accuracy(y_true, y_prob):
    assert y_true.ndim == 1 and y_true.size() == y_prob.size()
    y_prob = y_prob > 0.5
    return (y_true == y_prob).sum().item() / y_true.size(0)

def custom_tokenize(text):
    if not text:
        text = ''
    return remove_special_characters(text)

def n_words(sentence):
  return len(sentence.split(' '))

def find_indexes_to_drop(df):
  indexes = []
  sentences = set()
  for item,sentence in df['sentence'].items():
      sentence = str(sentence)
      if(n_words(sentence)<2):
        indexes.append(item)
        sentences.add(sentence)
  return indexes, sentences

(indexes, sentences) = find_indexes_to_drop(x_train_df)
x_train_df.drop(indexes, inplace=True)
y_train_df.drop(indexes, inplace=True)
x_train = x_train_df.to_numpy();
y_train = y_train_df.to_numpy();

# Doc2Vec alternative
## Train doc2vec model
doc2vec = Doc2Vec(tagged_data, vector_size = 20, window = 2, min_count = 1, epochs = 100)

#prepare data and embeddings, we want to prepare the data to transform it from strings and sentences to embeddings
newx_train = [doc2vec.infer_vector(word_tokenize(remove_special_characters(sentence[0]))) for sentence in x_train]
newx_test = [doc2vec.infer_vector(word_tokenize(remove_special_characters(sentence[0]))) for sentence in x_test]

for elem in newx_train[:50]:
  assert elem.shape==(20,)

x_train = torch.tensor(newx_train)
x_test = torch.tensor(newx_test)

In [ ]:
print("x_train->",x_train.shape)
print("x_test->",x_test.shape)
print("------")
print("y_train->",y_train.shape)
print("y_test->",y_test.shape)

In [ ]:
#the goal now is to build a pipeline where the words are tokenized and then fed into a classification model
#first, lets train the Doc2Vec model with our data

import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

def transform_data_for_test(data):
  new_data = data.float()
  return new_data.flatten()

class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.fc1 = nn.Linear(20,10)
    self.fc2 = nn.Linear(10,5)
    self.fc3 = nn.Linear(5,1)

  def forward(self, x):
      x = F.relu(self.fc1(x))
      x = F.relu(self.fc2(x))
      x = F.sigmoid(self.fc3(x))
      return x

model = Net()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

criterion = nn.BCELoss()
n_epochs = 100   # number of epochs to run
batch_size = 10000  # size of each batch
batch_start = torch.arange(0, len(x_train), batch_size)
best_mape = 100

losses = []

for epoch in range(n_epochs):
    model.train()
    for start in batch_start:
        # take a batch
        x_batch = torch.tensor(x_train[start:start+batch_size])
        y_batch = torch.tensor(y_train[start:start+batch_size])
        # forward pass
        y_pred = model(x_batch)
        loss = criterion(transform_data_for_test(y_pred), transform_data_for_test(y_batch))
        losses.append(loss.detach().numpy())
        # backward pass
        optimizer.zero_grad()
        loss.backward()
        # update weights
        optimizer.step()
    print("Epoch {}/{}, Loss: {:.3f}".format(epoch+1,n_epochs, loss))

    # evaluate accuracy at end of each epoch
    model.eval()
    y_pred = model(x_test)
    mape = criterion(transform_data_for_test(y_pred), transform_data_for_test(y_test))
    #print("error -> ", mape)
    if mape < best_mape:
        best_mape = mape
        best_weights = copy.deepcopy(model.state_dict())

# restore model and return best accuracy
# model.load_state_dict(best_weights)

plot(losses)

In [ ]:
print(len(losses))

Next steps

1. Try to implement cross validation with k-folds
2. Improve training data somehow
3. Create an algorithm to see which set of parameters best fit this model, data (batch size, traning of doc2vec, etc.)  


Main thing to do:
1. Improve results